# 05 — HuggingFace & Real GPT-2

> **Goal:** Use a real pretrained language model — the same way professional NLP engineers do.

### What changes from notebook 04?

| | mini-GPT (notebook 04) | GPT-2 (this notebook) |
|--|------------------------|----------------------|
| Built by | Us, from scratch | OpenAI (2019) |
| Vocabulary | 9 words | 50,257 words |
| Parameters | ~200 | 124 million |
| Training data | 5 sentences | 8 million web pages |
| Quality | basic | surprisingly good |

### The logic is identical — only the scale is different.


In [1]:
# Install HuggingFace Transformers
# (already installed in Colab, but run this to make sure)
!pip install transformers -q
print('Installation complete ✓')

Installation complete ✓


---
## Part 1 — Simplest Way: Pipeline

`pipeline` hides everything — tokenization, model, decoding.
You give text, you get text.


In [2]:
# ── Part 1: Pipeline ─────────────────────────────────────
from transformers import pipeline

# Load GPT-2 (downloads ~500MB first time, cached after)
print('Loading GPT-2... (first run takes ~1 minute)')
generator = pipeline(
    'text-generation',
    model='gpt2'
)
print('GPT-2 loaded ✓\n')

# Generate text
prompts = [
    'Artificial intelligence will',
    'Deep learning is',
    'The future of technology',
]

for prompt in prompts:
    result = generator(
        prompt,
        max_new_tokens=20,
        do_sample=True,
        temperature=0.8,
        pad_token_id=50256,
    )
    print(f"  Prompt: '{prompt}'")
    print(f"  Output: '{result[0]['generated_text']}'")
    print()

Loading GPT-2... (first run takes ~1 minute)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

...

GPT-2 loaded ✓



Both `max_new_tokens` (=20) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  Prompt: 'Artificial intelligence will'
  Output: 'Artificial intelligence will bring us together," said Michael R. Trenberth, a professor in the Department of Mechanical'



Both `max_new_tokens` (=20) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  Prompt: 'Deep learning is'
  Output: 'Deep learning is another area that seems to require a lot more effort and focus than traditional human language.

However'

  Prompt: 'The future of technology'
  Output: 'The future of technology

In June, Microsoft announced that the first Xbox 360 will support 3D motion capture in the'



---
## Part 2 — Tokenizer: The Same as Notebook 02

GPT-2's tokenizer does exactly what we built in notebook 02 — just bigger.

```
Our notebook 02:    word_to_idx['love'] = 7
GPT-2 tokenizer:    tokenizer.encode('love') = [1842]
```

In [3]:
# ── Part 2: Tokenizer Deep Dive ──────────────────────────
from transformers import GPT2Tokenizer

tokenizer = GPT2Tokenizer.from_pretrained('gpt2')

print(f'Vocabulary size: {tokenizer.vocab_size:,} words')
print(f'(Our notebook 02 had 9 words)\n')

# Show tokenization
examples = [
    'I love AI',
    'Deep learning',
    'unhappiness',       # rare word → split into pieces
    'ChatGPT',          # proper noun → split
]

print('Tokenization examples:')
print(f"  {'Text':<20} {'IDs':<25} {'Tokens'}")
print('  ' + '─' * 65)
for text in examples:
    ids    = tokenizer.encode(text)
    tokens = [tokenizer.decode([i]) for i in ids]
    print(f"  '{text}'")
    print(f"  {'':20} {str(ids):<25} {tokens}")
    print()

# Show encode → decode roundtrip
text  = 'Deep learning is amazing'
ids   = tokenizer.encode(text)
back  = tokenizer.decode(ids)
print(f'Roundtrip test:')
print(f"  '{text}' → {ids} → '{back}'")
print(f'  Matches original: {text == back} ✓')

Vocabulary size: 50,257 words
(Our notebook 02 had 9 words)

Tokenization examples:
  Text                 IDs                       Tokens
  ─────────────────────────────────────────────────────────────────
  'I love AI'
                       [40, 1842, 9552]          ['I', ' love', ' AI']

  'Deep learning'
                       [29744, 4673]             ['Deep', ' learning']

  'unhappiness'
                       [403, 71, 42661]          ['un', 'h', 'appiness']

  'ChatGPT'
                       [30820, 38, 11571]        ['Chat', 'G', 'PT']

Roundtrip test:
  'Deep learning is amazing' → [29744, 4673, 318, 4998] → 'Deep learning is amazing'
  Matches original: True ✓


---
## Part 3 — Model Forward Pass: The Same as Notebook 04

The model's forward pass is exactly what we built in notebook 04:
```
token IDs → embeddings → 12× attention+feedforward → logits → softmax → probabilities
```

In [4]:
# ── Part 3: Model Forward Pass ───────────────────────────
from transformers import GPT2LMHeadModel
import torch

model = GPT2LMHeadModel.from_pretrained('gpt2')
model.eval()

# Model info
total_params = sum(p.numel() for p in model.parameters())
print(f'GPT-2 Model Info:')
print(f'  Total parameters:   {total_params:,}')
print(f'  Transformer layers: {model.config.n_layer}')
print(f'  Attention heads:    {model.config.n_head}')
print(f'  Embedding dim:      {model.config.n_embd}')
print(f'  Max context length: {model.config.n_positions} tokens')
print()

# Forward pass
text   = 'Deep learning is'
inputs = tokenizer(text, return_tensors='pt')

with torch.no_grad():
    outputs = model(**inputs)

logits = outputs.logits
print(f'Input:  "{text}"')
print(f'Input IDs: {inputs["input_ids"][0].tolist()}')
print(f'Logits shape: {logits.shape}')
print(f'  = (batch=1, seq_len={logits.shape[1]}, vocab={logits.shape[2]:,})')

# Predict next word
last_logits = logits[0, -1, :]
probs = torch.softmax(last_logits, dim=-1)
top5  = torch.topk(probs, 5)

print(f'\nTop 5 predictions for next word after "{text}":')
print(f"  {'Word':<15} {'Probability'}")
print('  ' + '─' * 30)
for prob, idx in zip(top5.values, top5.indices):
    word = tokenizer.decode([idx])
    bar  = '█' * int(prob.item() * 50)
    print(f"  '{word}'{'':.<12} {prob.item():.3f}  {bar}")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


GPT-2 Model Info:
  Total parameters:   124,439,808
  Transformer layers: 12
  Attention heads:    12
  Embedding dim:      768
  Max context length: 1024 tokens

Input:  "Deep learning is"
Input IDs: [29744, 4673, 318]
Logits shape: torch.Size([1, 3, 50257])
  = (batch=1, seq_len=3, vocab=50,257)

Top 5 predictions for next word after "Deep learning is":
  Word            Probability
  ──────────────────────────────
  ' a'............ 0.152  ███████
  ' the'............ 0.076  ███
  ' an'............ 0.041  ██
  ' not'............ 0.036  █
  ' one'............ 0.019  


---
## Part 4 — Generation Strategies

Same as notebook 04 — different ways to pick the next word.

In [5]:
# ── Part 4: Generation Strategies ────────────────────────
tokenizer.pad_token = tokenizer.eos_token

prompt = 'Artificial intelligence will'
inputs = tokenizer(prompt, return_tensors='pt')

print(f'Prompt: "{prompt}"\n')

strategies = [
    {'label': 'Greedy (always most likely)',
     'params': {'do_sample': False}},
    {'label': 'Temperature=0.3 (conservative)',
     'params': {'do_sample': True, 'temperature': 0.3}},
    {'label': 'Temperature=0.8 (balanced)',
     'params': {'do_sample': True, 'temperature': 0.8}},
    {'label': 'Temperature=1.5 (creative)',
     'params': {'do_sample': True, 'temperature': 1.5}},
    {'label': 'Top-K=10 (pick from top 10)',
     'params': {'do_sample': True, 'top_k': 10}},
    {'label': 'Top-P=0.9 (nucleus sampling)',
     'params': {'do_sample': True, 'top_p': 0.9}},
]

for s in strategies:
    out = model.generate(
        inputs['input_ids'],
        max_new_tokens=12,
        pad_token_id=tokenizer.eos_token_id,
        **s['params']
    )
    text_out  = tokenizer.decode(out[0], skip_special_tokens=True)
    new_part  = text_out[len(prompt):]
    print(f"  [{s['label']}]")
    print(f"  ...{new_part}")
    print()

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Prompt: "Artificial intelligence will"

  [Greedy (always most likely)]
  ... be able to do things like search for and find people,

  [Temperature=0.3 (conservative)]
  ... be used to create a new type of human being, and

  [Temperature=0.8 (balanced)]
  ... be a big part of our future, but only if we

  [Temperature=1.5 (creative)]
  ... be so fast – and difficult to do without billions studying it

  [Top-K=10 (pick from top 10)]
  ... not always be able to solve the problem of how it will

  [Top-P=0.9 (nucleus sampling)]
  ... be increasingly useful for large corporations, such as those with billions



---
## Part 5 — Fine-Tuning on Custom Data

Fine-tuning = take a pretrained model + train a little more on your own data.

```
GPT-2 (general)  +  your data  =  specialized model
```

In [6]:
# ── Part 5: Fine-Tuning ───────────────────────────────────
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

# Training data — AI/ML sentences
training_texts = [
    'Deep learning is a subset of machine learning.',
    'Neural networks are inspired by the human brain.',
    'Artificial intelligence can solve complex problems.',
    'Transformers are the architecture behind modern LLMs.',
    'Attention mechanism allows models to focus on relevant words.',
    'Fine-tuning adapts a pretrained model to a specific task.',
    'GPT stands for Generative Pretrained Transformer.',
    'Training requires forward pass, loss, and backpropagation.',
]

# Setup tokenizer
tokenizer.pad_token = tokenizer.eos_token

# Dataset class
class TextDataset(Dataset):
    def __init__(self, texts, tokenizer, max_length=64):
        self.inputs = []
        for text in texts:
            enc = tokenizer(
                text,
                max_length=max_length,
                truncation=True,
                padding='max_length',
                return_tensors='pt',
            )
            self.inputs.append(enc['input_ids'].squeeze())

    def __len__(self):           return len(self.inputs)
    def __getitem__(self, idx):  return self.inputs[idx]


dataset    = TextDataset(training_texts, tokenizer)
dataloader = DataLoader(dataset, batch_size=2, shuffle=True)

print(f'Dataset: {len(dataset)} sentences')
print(f'Batches per epoch: {len(dataloader)}')

# Fine-tuning
ft_model   = GPT2LMHeadModel.from_pretrained('gpt2')
optimizer  = AdamW(ft_model.parameters(), lr=5e-5)
ft_model.train()

print('\n=== Fine-tuning ===\n')
print(f"  {'Epoch':<8} {'Loss'}")
print('  ' + '─' * 20)

for epoch in range(3):
    total_loss = 0
    for batch in dataloader:
        outputs = ft_model(input_ids=batch, labels=batch)
        loss    = outputs.loss
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(ft_model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
    avg = total_loss / len(dataloader)
    print(f'  {epoch+1:<8} {avg:.4f}')

print('\nFine-tuning complete ✓')

Dataset: 8 sentences
Batches per epoch: 4


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



=== Fine-tuning ===

  Epoch    Loss
  ────────────────────


`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


  1        5.0278
  2        0.7528
  3        0.5161

Fine-tuning complete ✓


In [7]:
# ── Compare: original vs fine-tuned ──────────────────────
ft_model.eval()

def generate_text(mdl, tkn, prompt, max_new_tokens=20):
    inputs = tkn(prompt, return_tensors='pt')
    with torch.no_grad():
        out = mdl.generate(
            inputs['input_ids'],
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.8,
            pad_token_id=tkn.eos_token_id,
        )
    return tkn.decode(out[0], skip_special_tokens=True)


test_prompts = ['Deep learning is', 'Neural networks']

print('=== Original GPT-2 ===')
orig_model = GPT2LMHeadModel.from_pretrained('gpt2')
orig_model.eval()
for p in test_prompts:
    out = generate_text(orig_model, tokenizer, p)
    print(f"  '{p}' →  {out[len(p):]}")

print('\n=== Fine-tuned on AI/ML data ===')
for p in test_prompts:
    out = generate_text(ft_model, tokenizer, p)
    print(f"  '{p}' →  {out[len(p):]}")

print('\nFine-tuned model talks more about AI/ML ✓')

=== Original GPT-2 ===


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  'Deep learning is' →   the best part of being an engineer. I don't have to be a computer, I can just
  'Neural networks' →  

As mentioned above, these networks are defined as being set-up by the user (i

=== Fine-tuned on AI/ML data ===
  'Deep learning is' →   a concept of machine learning.
  'Neural networks' →   are often used as a tool for developing models for complex networks.

Fine-tuned model talks more about AI/ML ✓


---
## Part 6 — Save and Load Your Model

In [8]:
# ── Save fine-tuned model ─────────────────────────────────
ft_model.save_pretrained('./my_gpt2_finetuned')
tokenizer.save_pretrained('./my_gpt2_finetuned')
print('Model saved to ./my_gpt2_finetuned/ ✓')
print()

# Load it back
loaded_model     = GPT2LMHeadModel.from_pretrained('./my_gpt2_finetuned')
loaded_tokenizer = GPT2Tokenizer.from_pretrained('./my_gpt2_finetuned')
loaded_model.eval()
print('Model loaded back successfully ✓')

# Test loaded model
out = generate_text(loaded_model, loaded_tokenizer, 'Deep learning')
print(f"\nLoaded model output: '{out}'")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to ./my_gpt2_finetuned/ ✓



Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Model loaded back successfully ✓

Loaded model output: 'Deep learning can also help to solve complex mathematical problems, but the biggest impact on how we use them is on'


---
## Summary


| Concept | Our code | HuggingFace equivalent |
|---------|----------|------------------------|
| Tokenize | `word_to_idx['love']` | `tokenizer.encode('love')` |
| Decode | `idx_to_word[7]` | `tokenizer.decode([1842])` |
| Forward pass | `forward(word_idx)` | `model(**inputs)` |
| Generate | `for step: predict next` | `model.generate()` |
| Fine-tune | training loop (notebook 01) | same loop with PyTorch |
| Save model | — | `model.save_pretrained()` |

**The difference is only scale — the logic is identical.**


---
**Next:** `06_finance_nlp.ipynb` — apply LLMs to real financial text analysis